# 02. Compare on dev and score on test

The configuration is chosen on dev only, then scored on test once. Test scores are saved, so later runs reuse them instead of scoring again.

In [ ]:
import os, subprocess, sys
from pathlib import Path

# Change this to your repo URL. For a private repo, add a GITHUB_TOKEN secret on Colab.
REPO_URL = "https://github.com/YOUR-USERNAME/vihsd-lora.git"

def github_token():
    if os.environ.get("GITHUB_TOKEN"):
        return os.environ["GITHUB_TOKEN"]
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None

def find_repo():
    places = [Path.cwd(), *Path.cwd().parents, Path("/content/vihsd-lora"), Path("/kaggle/working/vihsd-lora")]
    places += list(Path("/kaggle/input").glob("*")) + list(Path("/kaggle/input").glob("*/vihsd-lora"))
    return next((place for place in places if (place / "src" / "vihsd_lora").is_dir()), None)

repo = find_repo()
if repo is None:
    if "YOUR-USERNAME" in REPO_URL:
        raise ValueError("Set REPO_URL to your GitHub repo URL, then run this cell again.")
    repo = Path("/content" if Path("/content").exists() else "/kaggle/working") / "vihsd-lora"
    token = github_token()
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    subprocess.run(["git", "clone", "--depth", "1", url, str(repo)], check=True)
elif (repo / ".git").exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=False)
if not os.environ.get("VIHSD_SKIP_INSTALL"):
    # Colab ships an old torchao that makes peft fail when attaching LoRA. This repo does not use torchao, so remove it.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)
sys.path.insert(0, str(repo / "src"))
print("Repo:", repo)

In [ ]:
from vihsd_lora.config import Settings
from vihsd_lora.env import default_output_root, describe_gpu, detect_platform, free_space_gb, load_secrets

print("Platform:", detect_platform())
print("GPU:", describe_gpu())
print("Secrets found:", load_secrets())

settings = Settings.from_env(output_root=default_output_root())
free_gb = free_space_gb(settings.output_root)
print("Results are saved to:", settings.output_root)
print(f"Free space: {free_gb} GB")
if free_gb < 6:
    print("Warning: less than 6GB left. A full fine-tune run needs about 3GB, so clean up Drive and empty the trash.")

## Compare runs on dev

In [ ]:
from vihsd_lora.evaluate import compare_on_dev, evaluate_on_test, select_for_test

display(compare_on_dev(settings))
choice = select_for_test(settings)
print("Will score on test:", choice)

## Score on test

Only run this after all runs to compare have finished training.

In [ ]:
results = {name: evaluate_on_test(slug, settings) for name, slug in choice.items()}

## Compare LoRA with full fine-tune on the same test samples

If the confidence interval of the difference contains 0, there is not enough evidence to say which one is better.

In [ ]:
from vihsd_lora.report import compare_on_test

if {"lora", "full"} <= set(choice):
    print(compare_on_test(choice["lora"], choice["full"], settings))